#[실습]
1. 네이버 API로 블로그에서 '월드컵'에 대한 검색 결과 크롤링하여 파일로 저장
2. 공공API를 활용하여 도로명 주소조회 정보 저장 https://www.data.go.kr/data/15000124/openapi.do

In [ ]:
# 1. 네이버 API로 블로그에서 '월드컵'에 대한 검색 결과 크롤링 하여 파일로 저장

In [25]:
import os
import sys
import urllib.request
import datetime
import time
import json

## colab에서 secret키 사용하기 위해
from google.colab import userdata

# openapi 이용 예시
# https://github.com/naver/naver-openapi-guide

# developers naver에서 자신의 아이디와 비밀번호 발급받은 것 넣어줌
client_id = userdata.get('naver_client_id')
client_secret = userdata.get('naver_client_secret')

#[CODE 1]
def getRequestUrl(url):
    req = urllib.request.Request(url)
    req.add_header("X-Naver-Client-Id", client_id)
    req.add_header("X-Naver-Client-Secret", client_secret)

    try:
        response = urllib.request.urlopen(req)
        if response.getcode() == 200:
            print ("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode('utf-8') #검색결과 가져와서 한글파일로 읽어와라
    except Exception as e:
        print(e)
        print("[%s] Error for URL : %s" % (datetime.datetime.now(), url))
        return None

#[CODE2]
def getNaverSearch(node, srcText, start, display):
    base = "https://openapi.naver.com/v1/search"
    node = "/%s.json" % node
    parameters = "?query=%s&start=%s&display=%s" % (urllib.parse.quote(srcText), start, display)

    url = base + node + parameters
    responseDecode = getRequestUrl(url)   #[CODE 1]

    if (responseDecode == None):
        return None
    else:
        return json.loads(responseDecode)

#[CODE 3]
# 딕셔너리로 받은 데이터를 저장해준다.
def getPostData(post, jsonResult, cnt):
    title = post['title']
    description = post['description']
    link = post['link']

    pDate = datetime.datetime.strptime(post['postdate'],'%Y%m%d')
    pDate = pDate.strftime('%Y-%m-%d')

    jsonResult.append({'cnt':cnt, 'title':title, 'description': description,
   'link': link,   'pDate':pDate})
    return

#[CODE 0]
def main():
    node = 'blog'   # 크롤링 할 대상
    srcText = input('검색어를 입력하세요: ')
    cnt = 0
    jsonResult = []

    jsonResponse = getNaverSearch(node, srcText, 1, 100)  #[CODE 2]
    total = jsonResponse['total']

    while ((jsonResponse != None) and (jsonResponse['display'] != 0)):
        for post in jsonResponse['items']:
            cnt += 1
            getPostData(post, jsonResult, cnt)  #[CODE 3]

        start = jsonResponse['start'] + jsonResponse['display']
        if start == 1001 :
          break
        jsonResponse = getNaverSearch(node, srcText, start, 100)  #[CODE 2]


    print('전체 검색 : %d 건' %total)

    with open('%s_naver_%s.json' % (srcText, node), 'w', encoding='utf-8') as outfile:
        jsonFile = json.dumps(jsonResult,  indent=4, sort_keys=True,  ensure_ascii=False)

        outfile.write(jsonFile)

    print("가져온 데이터 : %d 건" %(cnt))
    print ('%s_naver_%s.json SAVED' % (srcText, node))

if __name__ == '__main__':
    main()

검색어를 입력하세요: 월드컵
[2025-02-05 08:13:36.420719] Url Request Success
[2025-02-05 08:13:37.782786] Url Request Success
[2025-02-05 08:13:38.991427] Url Request Success
[2025-02-05 08:13:40.203636] Url Request Success
[2025-02-05 08:13:41.404189] Url Request Success
[2025-02-05 08:13:42.600223] Url Request Success
[2025-02-05 08:13:43.831385] Url Request Success
[2025-02-05 08:13:45.026888] Url Request Success
[2025-02-05 08:13:46.400648] Url Request Success
[2025-02-05 08:13:47.589937] Url Request Success
전체 검색 : 2447121 건
가져온 데이터 : 1000 건
월드컵_naver_blog.json SAVED


In [ ]:
# 2. 공공API를 활용하여 도로명 주소조회 정보 저장 https://www.data.go.kr/data/15000124/openapi.do

In [30]:
!pip install xmltodict

In [37]:
import json
import requests
import xmltodict  # XML 데이터를 JSON으로 변환하기 위한 라이브러리


# API_KEY = 'VfpJWSbfEpMaKPKABH%2FePBaZJBuLIGNKaZHsPkDT40UlswButY7tmbTlnvjpht%2FntuQk861S2rdHnM602wek3w%3D%3D'
# API_KEY_decode = requests.utils.unquote(API_KEY)

# decoding이 된 api key
API_KEY = 'VfpJWSbfEpMaKPKABH/ePBaZJBuLIGNKaZHsPkDT40UlswButY7tmbTlnvjpht/ntuQk861S2rdHnM602wek3w=='

#1. searchSe = dong 인 경우 : 요청된 지번 주소에 대해 일치하는 도로명 주소와 지번주소를 표기
#2. searchSe = road 인 경우 : 요청된 도로명 주소에 대해 일치하는 도로명 주소와 지번주소를 표기

req_url = 'http://openapi.epost.go.kr/postal/retrieveNewAdressAreaCdService/retrieveNewAdressAreaCdService/getNewAddressListAreaCd'

searchSe = input("dong or road? : ")

srchwrd = input("검색 : ")

countPerPage = 10 # 한 페이지에 포함된 결과 수
countPage = 1 # 페이지 번호

params = {'ServiceKey' : API_KEY,
          'searchSe' : searchSe,
          'srchwrd' : srchwrd,
          'countPerPage' : countPerPage,'countPage' : countPage,
          'returnType': 'json'  # JSON 형식으로 요청
        }

r = requests.get(req_url, params = params)

# XML 데이터를 딕셔너리로 변환
try:
    dict_data = xmltodict.parse(r.text)

    # 변환된 딕셔너리 출력 (확인용)
    print("\nJSON 형식으로 변환된 데이터:")
    print(json.dumps(dict_data, indent=4, ensure_ascii=False))  # 한글 깨짐 방지

    # JSON 파일로 저장
    with open('address_data.json', 'w', encoding='utf-8') as json_file:
        json.dump(dict_data, json_file, ensure_ascii=False, indent=4)

    print("\nJSON 파일로 저장되었습니다: address_data.json")

except Exception as e:
    print("XML 파싱 중 오류 발생:", e)

# 못 가지고 오는 경우도 있기 때문에 나눠줌
if dict_data['NewAddressListResponse']['cmmMsgHeader']['successYN'] == 'Y' :
    post_code = dict_data['NewAddressListResponse']['newAddressListAreaCd']['zipNo']
    road_adr = dict_data['NewAddressListResponse']['newAddressListAreaCd']['lnmAdres']
    dong_adr = dict_data['NewAddressListResponse']['newAddressListAreaCd']['rnAdres']
    print("우편번호 :", post_code)
    print("도로명주소 :", road_adr )
    print("주소 :", dong_adr)
else:
    print("검색 결과가 없습니다.")

dong or road? : dong
검색 : 주월동 408-1

JSON 형식으로 변환된 데이터:
{
    "NewAddressListResponse": {
        "cmmMsgHeader": {
            "requestMsgId": null,
            "responseMsgId": null,
            "responseTime": "20250205:17294090",
            "successYN": "Y",
            "returnCode": "00",
            "errMsg": null,
            "totalCount": "1",
            "countPerPage": "10",
            "totalPage": "1",
            "currentPage": null
        },
        "newAddressListAreaCd": {
            "zipNo": "61725",
            "lnmAdres": "광주광역시 남구 서문대로 745 (주월동, 빅스포)",
            "rnAdres": "광주광역시 남구 주월동 408-1 빅스포"
        }
    }
}

JSON 파일로 저장되었습니다: address_data.json
우편번호 : 61725
도로명주소 : 광주광역시 남구 서문대로 745 (주월동, 빅스포)
주소 : 광주광역시 남구 주월동 408-1 빅스포
